# Limpieza de Datos — IPM Colombia

Transforma el archivo Excel oficial del DANE en un CSV limpio y estructurado para el dashboard de Pobreza Multidimensional.

**Pasos:**
1. Carga el Excel con MultiIndex de encabezados
2. Aplana las columnas a formato simple `Año_Categoría`
3. Convierte a formato largo con `melt()`
4. Limpia asteriscos y saltos de línea
5. Añade códigos DANE por departamento
6. Exporta a `../data/ipm_dpto.csv`

> **Requisito:** descarga el archivo `anex-PMultidimensional-Departamental-2025.xlsx`
> del portal del DANE y colócalo en la misma carpeta que este notebook.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from pathlib import Path
import pandas as pd

# El Excel debe estar en la misma carpeta que este notebook
EXCEL_PATH = Path("anex-PMultidimensional-Departamental-2025.xlsx")

df = pd.read_excel(
    EXCEL_PATH,
    sheet_name="IPM_Departamentos",
    header=[11, 12]
)

# Aplanar el MultiIndex de columnas
nuevas_columnas = []
for col in df.columns:
    nivel0 = str(col[0]).strip()
    nivel1 = str(col[1]).strip()
    if "Unnamed" in nivel1:
        nuevas_columnas.append(nivel0)
    else:
        nuevas_columnas.append(f"{nivel0}_{nivel1}")

df.columns = nuevas_columnas

print("Columnas después de aplanar:")
print(df.columns.tolist())

In [ ]:
# Eliminar filas vacías
df = df.dropna(subset=["Departamento"])

# Formato largo
df_long = df.melt(
    id_vars=["Departamento"],
    var_name="Año_Categoria",
    value_name="IPM"
)

# Separar Año y Categoría (n=1 para no cortar en espacios de la categoría)
df_long[["Año", "Categoria"]] = df_long["Año_Categoria"].str.split("_", n=1, expand=True)

# Limpiar asteriscos: "2020**" → "2020"
df_long["Año"] = df_long["Año"].str.replace(r"\*+", "", regex=True).str.strip()

# Limpiar salto de línea en categoría
df_long["Categoria"] = df_long["Categoria"].str.replace(r"\n", " ", regex=True)

# Eliminar columna auxiliar y reordenar
df_long = df_long.drop(columns=["Año_Categoria"])
df_long = df_long[["Departamento", "Año", "Categoria", "IPM"]]

# Quitar filas donde IPM no es número (filas de encabezados residuales)
df_long = df_long[pd.to_numeric(df_long["IPM"], errors="coerce").notna()]
df_long["IPM"] = df_long["IPM"].astype(float)
df_long["Año"] = df_long["Año"].astype(int)

print(df_long.head(12))
print(f"\nShape: {df_long.shape}")
print(f"\nDepartamentos únicos: {df_long['Departamento'].nunique()}")
print(f"Años únicos: {sorted(df_long['Año'].unique())}")

In [ ]:
codigos_dpto = {
    "Antioquia": "05",
    "Atlántico": "08",
    "Bogotá D.C.": "11",
    "Bolívar": "13",
    "Boyacá": "15",
    "Caldas": "17",
    "Caquetá": "18",
    "Casanare": "85",
    "Cauca": "19",
    "Cesar": "20",
    "Chocó": "27",
    "Córdoba": "23",
    "Cundinamarca": "25",
    "Guainía": "94",
    "Guaviare": "95",
    "Huila": "41",
    "La Guajira": "44",
    "Magdalena": "47",
    "Meta": "50",
    "Nariño": "52",
    "Norte de Santander": "54",
    "Putumayo": "86",
    "Quindío": "63",
    "Risaralda": "66",
    "San Andrés y Providencia": "88",
    "Santander": "68",
    "Sucre": "70",
    "Tolima": "73",
    "Valle del Cauca": "76",
    "Vaupés": "97",
    "Vichada": "99",
    "Arauca": "81",
    "San Andrés": "88",  
    "Amazonas": "91",
}

df_long["cod_dpto"] = df_long["Departamento"].map(codigos_dpto)

In [ ]:
df_long = df_long.rename(columns={"Departamento": "nombre_dpto"})

In [ ]:
print(df_long)

In [ ]:
# Guardar en la carpeta data/ del proyecto
output_path = Path("../data/ipm_dpto.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)
df_long.to_csv(output_path, index=False, encoding="utf-8-sig")
print("CSV guardado en:", output_path.resolve())

---
## Hoja 2 — Indicadores de Privación por Departamento

Procesa el Excel con las 15 privaciones del IPM desagregadas por departamento, año y zona.

**Fuente esperada en el Excel:** hoja con nombre de indicador o similar.
> Ajusta `sheet_name` si tu archivo tiene nombre diferente.

In [ ]:
from pathlib import Path
import pandas as pd

# Misma fuente Excel: ajusta la hoja según corresponda
EXCEL_PATH = Path("anex-PMultidimensional-Departamental-2025.xlsx")

# Leer la hoja de indicadores de privación
# Ajusta sheet_name, header y usecols según la estructura real del Excel
df_ind = pd.read_excel(
    EXCEL_PATH,
    sheet_name="IPM_Indicadores",  # <-- ajustar si el nombre es diferente
    header=[11, 12]
)

# Aplanar MultiIndex igual que en la hoja principal
nuevas_cols = []
for col in df_ind.columns:
    n0 = str(col[0]).strip()
    n1 = str(col[1]).strip()
    if "Unnamed" in n1:
        nuevas_cols.append(n0)
    else:
        nuevas_cols.append(f"{n0}_{n1}")
df_ind.columns = nuevas_cols

# Eliminar filas vacías
df_ind = df_ind.dropna(subset=["Departamento"])

# Detectar columnas de valor (todas las que no son Departamento ni Variable/Indicador)
id_cols = ["Departamento", "Variable"]  # ajustar si la col de indicador tiene otro nombre
value_cols = [c for c in df_ind.columns if c not in id_cols]

# Formato largo
df_ind_long = df_ind.melt(
    id_vars=id_cols,
    value_vars=value_cols,
    var_name="Año_Categoria",
    value_name="IPM"
)

# Separar Año y Categoría
df_ind_long[["Año", "Categoria"]] = df_ind_long["Año_Categoria"].str.split("_", n=1, expand=True)
df_ind_long["Año"] = df_ind_long["Año"].str.replace(r"\*+", "", regex=True).str.strip()
df_ind_long["Categoria"] = df_ind_long["Categoria"].str.replace(r"\n", " ", regex=True)

# Limpiar tipos
df_ind_long = df_ind_long.drop(columns=["Año_Categoria"])
df_ind_long = df_ind_long[pd.to_numeric(df_ind_long["IPM"], errors="coerce").notna()]
df_ind_long["IPM"] = df_ind_long["IPM"].astype(float)
df_ind_long["Año"] = df_ind_long["Año"].astype(int)

# Agregar códigos DANE
df_ind_long["cod_dpto"] = df_ind_long["Departamento"].map(codigos_dpto)
df_ind_long = df_ind_long.rename(columns={"Departamento": "nombre_dpto", "Variable": "Variable"})

print(df_ind_long.head())
print("Shape:", df_ind_long.shape)

# Exportar
out_ind = Path("../data/ipm_indicadores_dpto.csv")
out_ind.parent.mkdir(parents=True, exist_ok=True)
df_ind_long.to_csv(out_ind, index=False, encoding="utf-8-sig")
print("Guardado:", out_ind.resolve())

---
## Hoja 3 — IPM por Sexo del Jefe de Hogar

Procesa el Excel con el IPM desagregado por sexo del jefe de hogar,
para el análisis de brecha de género en el dashboard.

In [ ]:
# Leer la hoja de sexo del jefe de hogar
# Ajusta sheet_name, header y usecols según la estructura real del Excel
df_sex = pd.read_excel(
    EXCEL_PATH,
    sheet_name="IPM_Sexo",  # <-- ajustar si el nombre es diferente
    header=[11, 12]
)

# Aplanar MultiIndex
nuevas_cols_s = []
for col in df_sex.columns:
    n0 = str(col[0]).strip()
    n1 = str(col[1]).strip()
    if "Unnamed" in n1:
        nuevas_cols_s.append(n0)
    else:
        nuevas_cols_s.append(f"{n0}_{n1}")
df_sex.columns = nuevas_cols_s

# Eliminar filas vacías
df_sex = df_sex.dropna(subset=["Departamento"])

# Formato largo (id_vars incluye Departamento y Sexo si ya existe como columna)
id_cols_s = ["Departamento", "Sexo"]  # ajustar si la col de sexo tiene otro nombre
value_cols_s = [c for c in df_sex.columns if c not in id_cols_s]

df_sex_long = df_sex.melt(
    id_vars=id_cols_s,
    value_vars=value_cols_s,
    var_name="Año",
    value_name="Valor"
)

# Limpiar año
df_sex_long["Año"] = df_sex_long["Año"].str.replace(r"\*+", "", regex=True).str.strip()
df_sex_long = df_sex_long[pd.to_numeric(df_sex_long["Valor"], errors="coerce").notna()]
df_sex_long["Valor"] = df_sex_long["Valor"].astype(float)
df_sex_long["Año"] = df_sex_long["Año"].astype(int)

# Agregar códigos DANE
df_sex_long["cod_dpto"] = df_sex_long["Departamento"].map(codigos_dpto)
df_sex_long = df_sex_long.rename(columns={"Departamento": "nombre_dpto"})

print(df_sex_long.head())
print("Shape:", df_sex_long.shape)

# Exportar
out_sex = Path("../data/ipm_sexo_dpto.csv")
out_sex.parent.mkdir(parents=True, exist_ok=True)
df_sex_long.to_csv(out_sex, index=False, encoding="utf-8-sig")
print("Guardado:", out_sex.resolve())